# SQL Interview Study Notes for Data Engineers
*Spacious Study Guide Style — Clean, readable, and export‑ready (PDF/Word/Notion).*  
*Primary source: Interview Query — "Top SQL Interview Questions for Data Engineers: The Ultimate Guide"* citeturn7search8

---

## Table of Contents
- [Entry‑Level SQL Questions](#entry-level-sql-questions)
- [Mid‑Level SQL Questions](#mid-level-sql-questions)
- [Senior‑Level SQL Questions](#senior-level-sql-questions)
- [Database Design & Modeling Questions](#database-design--modeling-questions)
- [SQL ETL and Data Pipeline Interview Questions](#sql-etl-and-data-pipeline-interview-questions)
- [SQL Analytics and Reporting Problems](#sql-analytics-and-reporting-problems)
- [Advanced SQL: Big Data, Cloud, and Dialect Differences](#advanced-sql-big-data-cloud-and-dialect-differences)
- [Behavioral Data Engineering Interview Questions](#behavioral-data-engineering-interview-questions)
- [Common SQL Interview Mistakes & Troubleshooting](#common-sql-interview-mistakes--troubleshooting)
- [Salary Negotiation and Career Growth Tips for Data Engineers](#salary-negotiation-and-career-growth-tips-for-data-engineers)
- [SQL Interview Preparation Checklist](#sql-interview-preparation-checklist)
- [FAQs: SQL Interview Questions for Data Engineers](#faqs-sql-interview-questions-for-data-engineers)
- [Conclusion: Master SQL, Master the Data Engineer Interview](#conclusion-master-sql-master-the-data-engineer-interview)

---

## Entry‑Level SQL Questions

### 1) Find neighborhoods with 0 users
**What it tests:** LEFT JOIN, anti‑join logic, NULL handling.  
*Context adapted from Interview Query guide.* citeturn7search8

**Solution**
```sql
SELECT n.neighborhood_id, n.name
FROM neighborhoods n
LEFT JOIN users u
  ON n.neighborhood_id = u.neighborhood_id
WHERE u.user_id IS NULL;
```

**Lecture Notes**
- Prefer `LEFT JOIN` + `IS NULL` over `NOT IN` to avoid NULL pitfalls.
- Production use: detect orphaned dimensions or coverage gaps. citeturn7search8

---

### 2) 2nd highest salary in Engineering
**Solution**
```sql
WITH s AS (
  SELECT employee_id, salary,
         DENSE_RANK() OVER (ORDER BY salary DESC) AS rnk
  FROM employees
  WHERE department = 'Engineering'
)
SELECT employee_id, salary
FROM s
WHERE rnk = 2;
```
**Lecture Notes:** Use `DENSE_RANK` for ties; be explicit about tie policy. citeturn7search8

---

### 3) Count transactions with multiple filters
```sql
SELECT COUNT(*) AS txn_count
FROM transactions
WHERE payment_method = 'card'
  AND status = 'successful'
  AND txn_date BETWEEN '2024-01-01' AND '2024-01-31';
```
**Lecture Notes:** Consider `COUNT(DISTINCT txn_id)`; push filters down. citeturn7search8

---

### 4) Overall friend‑request acceptance rate
```sql
SELECT
  SUM(CASE WHEN accepted = 1 THEN 1 ELSE 0 END)::float
  / COUNT(*) AS acceptance_rate
FROM friend_requests;
```
**Lecture Notes:** Cast to float to avoid integer division. citeturn7search8

---

### 5) Average order value by gender
```sql
WITH order_value AS (
  SELECT o.order_id, u.gender,
         SUM(oi.price * oi.qty) AS order_amount
  FROM orders o
  JOIN order_items oi ON o.order_id = oi.order_id
  JOIN users u        ON o.user_id   = u.user_id
  GROUP BY o.order_id, u.gender
)
SELECT gender, AVG(order_amount) AS avg_order_value
FROM order_value
GROUP BY gender;
```
**Lecture Notes:** Handle unknown genders; two‑stage aggregation pattern. citeturn7search8

---

### 6) Apple‑platform actions in Top‑5 (Nov 2020)
```sql
WITH filtered AS (
  SELECT *
  FROM actions
  WHERE platform = 'Apple'
    AND action_date >= DATE '2020-11-01'
    AND action_date <  DATE '2020-12-01'
),
ranked AS (
  SELECT action_id,
         COUNT(*) AS action_count,
         DENSE_RANK() OVER (ORDER BY COUNT(*) DESC) AS rk
  FROM filtered
  GROUP BY action_id
)
SELECT action_id, action_count
FROM ranked
WHERE rk <= 5
ORDER BY action_count DESC, action_id;
```
**Lecture Notes:** Be explicit about tie handling with `DENSE_RANK`. citeturn7search8

---

### 7) Per‑SSID max packets in first 10 minutes of 2022‑01‑01
```sql
WITH pkt AS (
  SELECT ssid, device_id, COUNT(*) AS pkt_count
  FROM packet_logs
  WHERE ts >= TIMESTAMP '2022-01-01 00:00:00'
    AND ts <  TIMESTAMP '2022-01-01 00:10:00'
  GROUP BY ssid, device_id
)
SELECT ssid, MAX(pkt_count) AS max_packets
FROM pkt
GROUP BY ssid
ORDER BY ssid;
```
**Lecture Notes:** Half‑open intervals; index `(ssid, ts)` or `(ts, ssid)`. citeturn7search8

---

### 8) Top 5 projects by budget‑to‑employee ratio (dedupe)
```sql
SELECT p.project_id,
       p.budget / COUNT(DISTINCT ep.employee_id) AS budget_per_employee
FROM projects p
JOIN employee_projects ep ON p.project_id = ep.project_id
GROUP BY p.project_id
ORDER BY budget_per_employee DESC
LIMIT 5;
```
**Lecture Notes:** Use `COUNT(DISTINCT ...)` to handle duplicates robustly. citeturn7search8

---

## Mid‑Level SQL Questions

### 9) Top 3 departments by average salary
```sql
SELECT department, AVG(salary) AS avg_salary
FROM employees
GROUP BY department
ORDER BY avg_salary DESC
LIMIT 3;
```
**Lecture Notes:** Round for presentation; index on department. citeturn7search8

---

### 10) Current salary per employee (anomaly check)
```sql
-- open-ended to_date
SELECT employee_id, salary
FROM salaries
WHERE to_date = DATE '9999-01-01';

-- latest by timestamp
WITH latest AS (
  SELECT employee_id, salary,
         ROW_NUMBER() OVER (PARTITION BY employee_id ORDER BY updated_at DESC) AS rn
  FROM salaries
)
SELECT employee_id, salary
FROM latest
WHERE rn = 1;
```
**Lecture Notes:** Detect duplicates with window counts; list missing current rows with anti‑joins. citeturn7search8

---

### 11) Over‑budget project labeling
```sql
SELECT p.project_id,
       CASE WHEN COALESCE(e.total_spend, 0) > p.budget
            THEN 'overbudget' ELSE 'within budget' END AS status
FROM projects p
LEFT JOIN (
  SELECT project_id, SUM(amount) AS total_spend
  FROM expenditures
  GROUP BY project_id
) e ON p.project_id = e.project_id;
```
**Lecture Notes:** Use `COALESCE`; optionally return variance as diagnostic. citeturn7search8

---

### 12) % queries where all ratings < 3
```sql
SELECT
  COUNT(*) FILTER (WHERE max_rating < 3) * 1.0 / COUNT(*) AS pct_low_quality
FROM (
  SELECT query_id, MAX(rating) AS max_rating
  FROM search_results
  GROUP BY query_id
) q;
```
**Lecture Notes:** Group‑level predicate via `MAX` / `HAVING`. citeturn7search8

---

### 13) Top 5 by budget / employee count (safe division)
```sql
WITH emp_count AS (
  SELECT project_id, COUNT(DISTINCT employee_id) AS emp_cnt
  FROM employee_projects
  GROUP BY project_id
)
SELECT p.project_id,
       p.budget / NULLIF(e.emp_cnt, 0) AS budget_per_emp
FROM projects p
JOIN emp_count e USING (project_id)
ORDER BY budget_per_emp DESC NULLS LAST
LIMIT 5;
```
**Lecture Notes:** `NULLIF` avoids divide‑by‑zero; watch for small denominators. citeturn7search8

---

### 14) 2019 month‑over‑month revenue change
```sql
WITH rev AS (
  SELECT DATE_TRUNC('month', order_date) AS month,
         SUM(revenue) AS rev
  FROM orders
  WHERE order_date >= DATE '2019-01-01'
    AND order_date <  DATE '2020-01-01'
  GROUP BY 1
)
SELECT month,
       rev,
       LAG(rev) OVER (ORDER BY month) AS prev_rev,
       (rev - LAG(rev) OVER (ORDER BY month)) * 1.0 / LAG(rev) OVER (ORDER BY month) AS mom_change
FROM rev
ORDER BY month;
```
**Lecture Notes:** Handle first month null; consider calendar spine. citeturn7search8

---

### 15) Friends of a user who like a page
```sql
SELECT COUNT(DISTINCT f.friend_id) AS friend_likes
FROM friendships f
JOIN page_likes pl ON pl.user_id = f.friend_id
WHERE f.user_id = :user_id
  AND pl.page_id = :page_id;
```
**Lecture Notes:** DISTINCT prevents overcount in multi‑action logs. citeturn7search8

---

### 16) Top 3 earners per department
```sql
SELECT department_id, employee_id, salary
FROM (
  SELECT department_id, employee_id, salary,
         RANK() OVER (PARTITION BY department_id ORDER BY salary DESC) AS rnk
  FROM employees
) s
WHERE rnk <= 3
ORDER BY department_id, rnk, employee_id;
```
**Lecture Notes:** Clarify tie semantics (`RANK` vs `DENSE_RANK`). citeturn7search8

---

### 17) Jan‑2020 histogram of comments per user
```sql
WITH cnt AS (
  SELECT user_id, COUNT(*) AS comments
  FROM comments
  WHERE comment_date >= DATE '2020-01-01'
    AND comment_date <  DATE '2020-02-01'
  GROUP BY user_id
)
SELECT comments AS bucket, COUNT(*) AS users_in_bucket
FROM cnt
GROUP BY comments
ORDER BY comments;
```
**Lecture Notes:** Generate missing bins using a numbers table or `generate_series`. citeturn7search8

---

### 18) Score closest to a SAT benchmark
```sql
SELECT student_id, score
FROM sat_scores
ORDER BY ABS(score - :benchmark) ASC, student_id ASC
LIMIT 1;
```
**Lecture Notes:** Deterministic secondary ordering avoids random ties. citeturn7search8

---

## Senior‑Level SQL Questions

### 19) CTR across queries (optimized)
```sql
WITH agg AS (
  SELECT query_norm,
         COUNT(*) FILTER (WHERE event = 'impression') AS impressions,
         COUNT(*) FILTER (WHERE event = 'click')      AS clicks
  FROM search_events
  GROUP BY query_norm
)
SELECT query_norm,
       CASE WHEN impressions = 0 THEN NULL
            ELSE clicks * 1.0 / impressions
       END AS ctr
FROM agg
ORDER BY ctr DESC NULLS LAST, query_norm;
```
**Lecture Notes:** Pre‑aggregate; composite indexes; consider MV/caching. citeturn7search8

---

### 20) Cumulative users with monthly resets
```sql
SELECT created_at::date AS day,
       SUM(1) OVER (
         PARTITION BY DATE_TRUNC('month', created_at)
         ORDER BY created_at
         ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
       ) AS cumulative_users
FROM users;
```
**Lecture Notes:** Verify window frames; partition boundaries by month. citeturn7search8

---

### 21) Average response time to system messages
```sql
WITH sys AS (
  SELECT thread_id, timestamp AS sys_ts
  FROM messages
  WHERE message_type = 'system'
),
resp AS (
  SELECT m.thread_id, m.timestamp AS user_ts
  FROM messages m
  WHERE m.message_type = 'user'
),
paired AS (
  SELECT s.thread_id,
         MIN(r.user_ts) AS first_user_after_system
  FROM sys s
  JOIN resp r ON r.thread_id = s.thread_id AND r.user_ts > s.sys_ts
  GROUP BY s.thread_id, s.sys_ts
)
SELECT AVG(EXTRACT(EPOCH FROM (first_user_after_system - sys_ts))) AS avg_response_secs
FROM paired;
```
**Lecture Notes:** Handle missing responses; index `(thread_id, timestamp)`. citeturn7search8

---

### 22) Daily minutes in flight per plane
```sql
SELECT plane_id,
       flight_date,
       SUM(EXTRACT(EPOCH FROM (arrival_ts AT TIME ZONE 'UTC' - departure_ts AT TIME ZONE 'UTC')) / 60) AS minutes_in_flight
FROM flights
GROUP BY plane_id, flight_date
ORDER BY plane_id, flight_date;
```
**Lecture Notes:** Normalize time zones; define overnight flight policy. citeturn7search8

---

### 23) Second‑longest flight per city pair
```sql
WITH normalized AS (
  SELECT LEAST(city_from, city_to) AS city_a,
         GREATEST(city_from, city_to) AS city_b,
         EXTRACT(EPOCH FROM (arrival_ts - departure_ts)) / 60 AS duration_min
  FROM flights
),
ranked AS (
  SELECT city_a, city_b, duration_min,
         ROW_NUMBER() OVER (PARTITION BY city_a, city_b ORDER BY duration_min DESC) AS rn
  FROM normalized
)
SELECT city_a, city_b, duration_min
FROM ranked
WHERE rn = 2
ORDER BY city_a, city_b;
```
**Lecture Notes:** Normalize A‑B/B‑A; `ROW_NUMBER` gives exactly one row. citeturn7search8

---

## Database Design & Modeling Questions

> **What interviewers probe:** normalization vs. denormalization, dimensional modeling (star/snowflake), keys & constraints, indexing strategy, partitioning & clustering, SCD patterns, and schema evolution. *(Topics outlined in the Interview Query guide’s design/architecture emphasis.)* citeturn7search8

**Sample prompts & model answers**
- **When would you denormalize a warehouse schema?**  
  *Answer:* To reduce costly joins and speed up read‑heavy analytics, especially for stable dimensions. Trade off write complexity, storage, and potential inconsistency for faster reads. Use materialized views or dbt snapshots for control. citeturn7search8
- **Star vs. Snowflake: which and why?**  
  *Answer:* Star is simpler and faster for BI; snowflake saves storage and improves dimension reusability. Choose based on query patterns, BI tool behavior, and team skill set. citeturn7search8
- **SCD Type‑2 design in SQL?**  
  *Answer:* Use surrogate keys and validity ranges (`valid_from`, `valid_to`) with a “current” flag; enforce no overlaps; create UNIQUE constraint on `(business_key, valid_from)`. citeturn7search8
- **Partitioning & clustering strategy?**  
  *Answer:* Partition large fact tables by time; cluster on high‑selectivity columns used in filters/joins to improve pruning. Balance small‑file problems vs. partition explosion. citeturn7search8

**Design checklist**
- Choose keys (PK/FK) and constraints for integrity.  
- Favor **surrogate keys** for dimensions; stable business keys as natural keys.  
- Plan **indexes** for critical joins/filters.  
- Decide on **partitioning** and **clustering** based on access patterns.  
- Establish **SCD strategy** and schema‑migration process. citeturn7search8

---

## SQL ETL and Data Pipeline Interview Questions

> **What interviewers probe:** SQL in ELT/ETL steps, idempotency, late‑arriving data, deduping, CDC, orchestration dependencies, and quality checks. *(Aligned with scenario‑based sections in the guide.)* citeturn7search8

**Common tasks & patterns**
- **Incremental loads:** Use watermarks (`updated_at`), keep a high‑water table, and write `MERGE`/`UPSERT` statements with dedupe on `(pk, updated_at)`.
- **Idempotency:** Stage to temp tables; commit via `INSERT ... SELECT` or `MERGE`; wrap in transactions.
- **Late‑arriving facts:** Recompute aggregates with windowed reprocessing; keep audit columns (`ingested_at`).
- **Data quality:** Use NOT NULL/CHK constraints, uniqueness tests, and aggregation reconciliations. citeturn7search8

**Example: UPSERT with MERGE (ANSI‑ish)**
```sql
MERGE INTO target t
USING staging s
  ON t.id = s.id
WHEN MATCHED THEN
  UPDATE SET colA = s.colA, updated_at = s.updated_at
WHEN NOT MATCHED THEN
  INSERT (id, colA, updated_at)
  VALUES (s.id, s.colA, s.updated_at);
```
*Tip:* Adjust syntax for Snowflake/BigQuery/SQL Server dialects. citeturn7search8

---

## SQL Analytics and Reporting Problems

> **What interviewers probe:** Window functions, cohorting, sessionization, conditional aggregations, and KPI design. *(Reflected across the guide’s coding & scenario questions.)* citeturn7search8

**Patterns**
- **Cohorts / retention:** `DATE_TRUNC` to cohort by signup month; join to events and compute retained counts with `COUNT(DISTINCT user_id)` and time offsets.
- **Sessionization:** `SUM( CASE WHEN gap > 30min THEN 1 ELSE 0 END )` over ordered events to assign session IDs.
- **Percentiles:** Use `PERCENTILE_CONT`/`APPROX_QUANTILES` by dialect for P90/P95 latency. citeturn7search8

**Template: Running totals with resets**
```sql
SELECT d, SUM(metric) OVER (
  PARTITION BY DATE_TRUNC('month', d)
  ORDER BY d
  ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
) AS month_running
FROM daily_metrics;
```
*Explains KPI windows without cross‑month bleed.* citeturn7search8

---

## Advanced SQL: Big Data, Cloud, and Dialect Differences

> **What interviewers probe:** Warehouse features (Snowflake, BigQuery, Redshift), partition pruning, cost control, UDFs, semi‑structured types, and dialect quirks. *(Advanced topics are highlighted in the guide.)* citeturn7search8

**Key differences & reminders**
- **Snowflake:** `VARIANT` for JSON, clustering keys, time‑travel; `MERGE` supported.  
- **BigQuery:** Default standard SQL, partitioned/clustered tables, per‑query bytes scanned; use `APPROX_*` functions.  
- **Redshift:** Dist/Sort keys, spectrum for external data, late‑binding views; `VACUUM/ANALYZE` maintenance.  
- **Spark SQL:** Lazy evaluation; watch for shuffle; optimize with `REPARTITION` and `BROADCAST`. citeturn7search8

**Cost & performance tips**
- Push down predicates; select only needed columns.  
- Use partition filters to prune; cache hot aggregates; pre‑compute via materialized views or scheduled jobs. citeturn7search8

---

## Behavioral Data Engineering Interview Questions

> **What interviewers probe:** collaboration, incident response, decision trade‑offs, and communication with non‑technical stakeholders. *(The guide emphasizes soft‑skills alongside SQL.)* citeturn7search8

**Sample prompts (STAR‑ready)**
- **Tell me about a pipeline you stabilized under time pressure.** Focus on detection, rollback, RCA, and prevention.
- **Describe a time you chose denormalization over normalization.** Outline trade‑offs and validation plan.
- **How do you collaborate with analysts on schema design?** Emphasize empathy, documentation, and versioning. citeturn7search8

---

## Common SQL Interview Mistakes & Troubleshooting

- Ignoring **NULL** semantics → wrong counts/filters (`= NULL` instead of `IS NULL`).  
- Using `NOT IN` with NULL‑containing subqueries → unexpected empty results.  
- Forgetting **division by zero** guards (`NULLIF`).  
- Missing **tie** policy with rankings (`RANK` vs `DENSE_RANK` vs `ROW_NUMBER`).  
- No **deterministic ORDER BY** on LIMIT queries.  
- Skipping **EXPLAIN** / not reasoning about indexes.  
- Not validating with **unit checks** (row counts before/after, sums vs. source). citeturn7search8

**Quick debug checklist**
- Reproduce with a **minimal dataset**.  
- Check **join cardinality** and unintended cross‑joins.  
- Validate **filters** and **time zones**.  
- Compare **counts** at each step (staging → final). citeturn7search8

---

## Salary Negotiation and Career Growth Tips for Data Engineers

> The guide discusses role expectations and growth themes around data engineering interviews; here’s a concise, actionable take tailored for SQL‑heavy roles. citeturn7search8

**Negotiation**
- Benchmark via levels and local market; bring **evidence** (offers, salary surveys).  
- Negotiate total comp: base, bonus, equity, relocation, training budget.  
- Time your ask **after** strong signals (onsite success / verbal). citeturn7search8

**Growth**
- Own **data products** end‑to‑end; document lineage and SLOs.  
- Invest in **performance and cost** literacy on your warehouse.  
- Mentor juniors; champion **testing and observability** in SQL pipelines. citeturn7search8

---

## SQL Interview Preparation Checklist

- **Core SQL:** Joins, grouping, filtering, window functions, CTEs.  
- **Modeling:** Star vs snowflake, SCD, keys & constraints.  
- **ETL/ELT:** Incrementals, idempotency, CDC, quality checks.  
- **Analytics:** Cohorts, sessionization, percentiles, time windows.  
- **Performance:** Indexes, partitioning, clustering, EXPLAIN plans.  
- **Platforms:** Snowflake/BigQuery/Redshift basics; dialect differences.  
- **Behavioral:** Incident stories, trade‑off decisions, stakeholder comms.  
- **Practice:** Re‑implement solutions from this sheet *without* notes; time yourself. citeturn7search8

---

## FAQs: SQL Interview Questions for Data Engineers

**Q1. How much SQL vs. Python in data engineer interviews?**  
A: Expect at least one **SQL‑heavy** round and a Python/coding round; senior loops add system design and data modeling. citeturn7search8

**Q2. Do warehouses count as "SQL" questions?**  
A: Yes—Snowflake/BigQuery/Redshift all use SQL dialects; be ready for cost/perf questions and `MERGE`/partition patterns. citeturn7search8

**Q3. Should I memorize syntax for every dialect?**  
A: Focus on **portable patterns**, then learn key dialect differences (e.g., `QUALIFY` in BigQuery, `SAMPLE` in Snowflake). citeturn7search8

---

## Conclusion: Master SQL, Master the Data Engineer Interview

Strong SQL is the backbone of **data modeling, ETL/ELT, analytics, and pipeline debugging**. Mastering query correctness *and* performance—plus clear communication—lets you shine across rounds from entry to senior. Use these notes to drill patterns, then practice on real, messy datasets. *(Themes and question styles aligned with the Interview Query guide.)* citeturn7search8

---

*Prepared for: Study and interview practice (Data Engineering SQL).*  
*Author: Generated by M365 Copilot*  
*Last updated: 2026‑03‑21*
